# NB08: General Essentiality Model — **Exploratory / Not Part of Submitted Pipeline**

This notebook trains a single multi-label CatBoost model that uses `stressor` as a categorical feature alongside protein sequence features, enabling a general-purpose essentiality classifier across all stressors simultaneously. It is **exploratory** and is **not executed or required** for the main analysis:

- Core findings rely on the per-stressor CatBoostRegressor models in NB05 and the candidate ranking in NB09.
- NB08 uses a different paradigm (one joint model vs. 11 independent models) and subsamples training data to keep memory tractable.
- Results here would supplement — not replace — the per-stressor regression approach.

**To run**: execute NB05 first to generate feature parquets, then run this notebook (no Spark required; ~4 GB RAM with default subsampling).

In [ ]:
import sys, json, logging, pickle, time
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef, precision_recall_curve)
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models' / 'general'
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
CONFIG = {
    'SEED': 42, 'TEST_SIZE': 0.2, 'CALIBRATION_SIZE': 0.1,
    'CATBOOST_ITERATIONS': 1000, 'CATBOOST_LEARNING_RATE': 0.03, 'CATBOOST_DEPTH': 6,
    'SUBSAMPLE_PER_STRESSOR': 5000
}

In [ ]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

In [ ]:
# Build feature matrix
X_list = [pd.read_parquet(DATA_DIR / f"features_{n}.parquet").drop(columns=['organism'], errors='ignore')
          for n in best_combo]
X_full = pd.concat(X_list, axis=1)

In [ ]:
# Melt labels: one row per (protein, stressor)
records = []
for s in active_stressors:
    if s not in labeled_pd.columns: continue
    sub = labeled_pd[[s,'organism']].rename(columns={s:'essential'})
    sub['stressor'] = s
    sub = sub.reset_index()   # protein_id as column
    n_sample = min(len(sub), CONFIG['SUBSAMPLE_PER_STRESSOR'])
    sub = sub.sample(n=n_sample, random_state=CONFIG['SEED'])
    records.append(sub)
all_labels = pd.concat(records, ignore_index=True)

X_reset = X_full.reset_index()
train_data = all_labels.merge(X_reset, on='protein_id', how='inner')
assert 'stressor' in train_data.columns, "stressor missing"

meta_cols = ['protein_id','organism','essential']
feature_cols = [c for c in train_data.columns if c not in meta_cols]

groups = train_data['organism']
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['SEED'])
tr_idx, te_idx = next(gss.split(train_data, train_data['essential'], groups=groups))

X_tr = train_data.iloc[tr_idx][feature_cols].reset_index(drop=True)
X_te = train_data.iloc[te_idx][feature_cols].reset_index(drop=True)
y_tr = train_data.iloc[tr_idx]['essential'].reset_index(drop=True)
y_te = train_data.iloc[te_idx]['essential'].reset_index(drop=True)
stressor_te = X_te['stressor']

g_tr = train_data.iloc[tr_idx]['organism'].reset_index(drop=True)
gss_cal = GroupShuffleSplit(n_splits=1, test_size=CONFIG['CALIBRATION_SIZE'], random_state=CONFIG['SEED'])
sub_idx, cal_idx = next(gss_cal.split(X_tr, y_tr, groups=g_tr))
X_sub, X_cal = X_tr.iloc[sub_idx], X_tr.iloc[cal_idx]
y_sub, y_cal = y_tr.iloc[sub_idx], y_tr.iloc[cal_idx]

In [ ]:
model = CatBoostClassifier(iterations=CONFIG['CATBOOST_ITERATIONS'],
                           learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
                           depth=CONFIG['CATBOOST_DEPTH'],
                           cat_features=['stressor'], eval_metric='F1',
                           random_seed=CONFIG['SEED'], verbose=100)
model.fit(X_sub, y_sub, verbose=100)

y_proba = model.predict_proba(X_te)[:,1]
y_pred = model.predict(X_te)

cal_proba = model.predict_proba(X_cal)[:,1]
platt = LogisticRegression().fit(cal_proba.reshape(-1,1), y_cal)
y_proba_cal = platt.predict_proba(y_proba.reshape(-1,1))[:,1]

In [ ]:
# Adaptive thresholds
thresholds = {}
for s in active_stressors:
    mask = stressor_te == s
    if mask.sum() < 20: continue
    y_s = y_te[mask].values
    y_p = y_proba[mask]
    prec, rec, thresh = precision_recall_curve(y_s, y_p)
    f1s = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-10)
    thresholds[s] = thresh[np.argmax(f1s)]

y_pred_adapt = np.zeros(len(y_te), dtype=int)
for s, th in thresholds.items():
    mask = stressor_te == s
    y_pred_adapt[mask] = (y_proba[mask] >= th).astype(int)

log.info(f"Overall AUC: {roc_auc_score(y_te, y_proba):.4f}")
log.info(f"Adaptive MCC: {matthews_corrcoef(y_te, y_pred_adapt):.4f}")

In [ ]:
# Save
model.save_model(str(MODEL_DIR / "general_essentiality.cbm"))
with open(MODEL_DIR / "general_essentiality_platt.pkl", 'wb') as f:
    pickle.dump(platt, f)
with open(MODEL_DIR / "adaptive_thresholds.json", 'w') as f:
    json.dump(thresholds, f)